In [20]:
# 파이썬 유틸리티 함수 정의
import matplotlib.pyplot as plt
import numpy as np
from skimage.io import imread
from skimage.color import rgb2gray, rgba2rgb

def plot_image(image, title=''):
    plt.imshow(image)
    plt.axis('off')
    plt.title(title, size=20)

def gaussian(M, std, sym=True):
    if M < 1:
        return np.array([])
    if M == 1:
        return np.ones(1, 'd')
    odd = M % 2
    if not sym and not odd:
        M = M + 1
    n = np.arange(0, M) - (M - 1.0) / 2.0
    sig2 = 2 * std * std
    w = np.exp(-n ** 2 / sig2)
    if not sym and not odd:
        w = w[:-1]
    return w

In [ ]:
import numpy.fft as fp
import numpy as np
from scipy import signal

imSrc = np.mean(imread('lena.jpg'), axis=2)
gaussKernel = np.outer(gaussian(imSrc.shape[0], 5), gaussian(imSrc.shape[1], 5))

freqSrc = fp.fft2(imSrc)
freqKernel = fp.fft2(fp.ifftshift(gaussKernel))

imConv = freqSrc * freqKernel
imResult = fp.ifft2(imConv).real

plt.figure(figsize=(20,15))
plt.gray()

plt.subplot(2,3,1), plot_image(imSrc, 'Original Image')
plt.subplot(2,3,2), plot_image(gaussKernel, 'Gaussian Kernel')
plt.subplot(2,3,3), plot_image(imResult, 'Output Image')

plt.subplot(2,3,4), plot_image(20*np.log10( 0.1 + np.abs(fp.fftshift(freqSrc))), 'Original Fourier Spectrum')
plt.subplot(2,3,5), plot_image(20*np.log10( 0.1 + np.abs(fp.fftshift(freqKernel))), 'Gaussian Fourier Spectrum')
plt.subplot(2,3,6), plot_image(20*np.log10( 0.1 + np.abs(fp.fftshift(imConv))), 'Output Fourier Spectrum')

plt.show()

In [ ]:
srcImg = np.mean(imread('rhino.jpg'), axis=2)

freqDomain = fp.fft2(srcImg)
freqFilter = fp.fftshift(freqDomain)
freqTmp = np.copy(freqFilter)

(w, h) = freqDomain.shape
hW, hH = int(w/2), int(h/2)

# HPF
freqFilter[hW-10:hW+10,hH-10:hH+10] = 0

# LPF
#freqTmp[hW-10:hW+10,hH-10:hH+10] = 0
#freqFilter -= freqTmp

resImg = np.clip(fp.ifft2(fp.ifftshift(freqFilter)).real, 0, 255)

plt.figure(figsize=(20,15))
plt.gray()

plt.subplot(2,2,1), plot_image(srcImg, 'Original Image')
plt.subplot(2,2,2), plot_image(resImg, 'Output Image')

plt.subplot(2,2,3), plot_image(20*np.log10( 0.1 + np.abs(fp.fftshift(freqDomain))), 'Original Fourier Spectrum')
plt.subplot(2,2,4), plot_image(20*np.log10( 0.1 + np.abs(freqFilter)), 'Filtered Fourier Spectrum')

plt.show()